In [3]:
PREFIX = "https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/02-vector-search/embed"
!wget {PREFIX}/download.py
!wget {PREFIX}/embedder.py

--2026-06-20 14:58:59--  https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/02-vector-search/embed/download.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.109.133, 185.199.110.133, 185.199.108.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.109.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1376 (1.3K) [text/plain]
Saving to: ‘download.py’

download.py         100%[===================>]   1.34K  --.-KB/s    in 0s      

2026-06-20 14:58:59 (66.9 MB/s) - ‘download.py’ saved [1376/1376]



--2026-06-20 14:59:00--  https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/02-vector-search/embed/embedder.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1520 (1.5K) [text/plain]
Saving to: ‘embedder.py’

embedder.py         100%[===================>]   1.48K  --.-KB/s    in 0s      

2026-06-20 14:59:00 (12.9 MB/s) - ‘embedder.py’ saved [1520/1520]



In [1]:
!uv run python download.py

  exists models/Xenova/all-MiniLM-L6-v2/tokenizer.json
  exists models/Xenova/all-MiniLM-L6-v2/model.onnx


In [2]:
from embedder import Embedder

2026-06-22 23:01:43.274361450 [W:onnxruntime:Default, device_discovery.cc:133 GetPciBusId] Skipping pci_bus_id for PCI path at "/sys/devices/LNXSYSTM:00/LNXSYBUS:00/PNP0A03:00/device:07/VMBUS:01/5620e0c7-8062-4dce-aeb7-520c7ef76171" because filename "5620e0c7-8062-4dce-aeb7-520c7ef76171" did not match expected pattern of [0-9a-f]+:[0-9a-f]+:[0-9a-f]+[.][0-9a-f]+


# Q1. Embedding a query

Embed the following query:

    How does approximate nearest neighbor search work?

The embedder returns a vector of 384 numbers. What's the first value (v[0])?

## Answer:
-0.02


In [3]:
model = Embedder()
q1 = "How does approximate nearest neighbor search work?"
v1 = model.encode(q1)
print(v1[0])

-0.02058203437252893


# Q2. Cosine similarity

The embedder returns normalized vectors, so the dot product between two of them is their cosine similarity.

Take the page 02-vector-search/lessons/07-sqlitesearch-vector.md, embed its content, and compute the cosine similarity with the query vector from Q1. What do you get?
## Answer: 0.37

In [7]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

documents = [file.parse() for file in reader.read()]

In [8]:
# Each document is a dictionary with a filename and content, and there are 72 pages.
# Take the page 02-vector-search/lessons/07-sqlitesearch-vector.md, embed its content, and compute the cosine similarity with the query vector from Q1. What do you get?
doc = next(doc for doc in documents if doc["filename"] == "02-vector-search/lessons/07-sqlitesearch-vector.md")
doc_vector = model.encode(doc["content"])
v1.dot(doc_vector)

np.float64(0.36107027225589694)

# Q3. Chunking and search by hand

Which file does the highest-scoring chunk belong to (its filename)?
## Answer:
'02-vector-search/lessons/07-sqlitesearch-vector.md'

In [11]:
# Q3. Chunking and search by hand

# A full page covers several topics, which waters down its embedding.

# We chunk the pages the same way as in homework 1:

# from gitsource import chunk_documents
# chunks = chunk_documents(documents, size=2000, step=1000)

# We embed every chunk's content with encode_batch, stack the vectors into a matrix X, and score the Q1 query against all chunks:

# scores = X.dot(v)
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)
X = model.encode_batch([chunk["content"] for chunk in chunks])



In [15]:
scores = X.dot(v1)
best = scores.argmax()
chunks[best]["filename"]

'02-vector-search/lessons/07-sqlitesearch-vector.md'

# Q4. Vector search with minsearch

We've done vector search by hand, which is good for learning, but it's not what we do in practice. In practice we use libraries.

Let's use VectorSearch from minsearch and run a search for the following query:

    What metric do we use to evaluate a search engine?

Which file is the filename of the first result?
## Answer:
    04-evaluation/lessons/05-search-metrics.md



In [ ]:
from minsearch import VectorSearch

vindex = VectorSearch(keyword_fields=["What metric do we use to evaluate a search engine?"])
vindex.fit(X, documents)

In [17]:
from minsearch import VectorSearch

vindex = VectorSearch()
vindex.fit(X, chunks)

q = "What metric do we use to evaluate a search engine?"
q_vec = model.encode(q)

results = vindex.search(q_vec, num_results=5)
results[0]["filename"]

'04-evaluation/lessons/05-search-metrics.md'

# Q5. Text search vs vector search

Vector search matches by meaning, keyword search by exact words.

Let's compare them. Index the same chunks with Index from minsearch. Use content as a text field.

Run both searches for this query:

    How do I store vectors in PostgreSQL?

Take the top 5 results from each method. Which file shows up in the vector results but not in the text results?

## Answer:
    02-vector-search/lessons/08-pgvector.md

In [18]:
text_index = minsearch.Index(text_fields=["content"])
text_index.fit(chunks)

q = "How do I store vectors in PostgreSQL?"

text_results = text_index.search(q, num_results=5)

q_vec = model.encode(q)
vector_results = vindex.search(q_vec, num_results=5)

text_files = {r["filename"] for r in text_results}
vector_files = {r["filename"] for r in vector_results}

print(vector_files - text_files)

{'02-vector-search/lessons/08-pgvector.md'}


In [19]:
print("text top-5:  ", text_files)
print("vector top-5:", vector_files)
print("shared:      ", text_files & vector_files)
print("vector only: ", vector_files - text_files)
print("text only:   ", text_files - vector_files)

text top-5:   {'03-orchestration/lessons/05-rag.md', '02-vector-search/lessons/01-intro.md', '02-vector-search/lessons/02-embeddings.md'}
vector top-5: {'03-orchestration/lessons/05-rag.md', '02-vector-search/lessons/08-pgvector.md'}
shared:       {'03-orchestration/lessons/05-rag.md'}
vector only:  {'02-vector-search/lessons/08-pgvector.md'}
text only:    {'02-vector-search/lessons/01-intro.md', '02-vector-search/lessons/02-embeddings.md'}


## Q6. Hybrid search
Run the query "How do I give the model access to tools?" with vector and text search and fuse the results with rrf:

results = rrf([vector_results, text_results])

Which file is ranked first after RRF?
## Answer: 
'01-agentic-rag/lessons/13-function-calling.md'


In [20]:
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]

In [21]:
q = "How do I give the model access to tools?"

q_vec = model.encode(q)
vector_results = vindex.search(q_vec, num_results=5)
text_results = text_index.search(q, num_results=5)

results = rrf([vector_results, text_results])
results[0]["filename"]

'01-agentic-rag/lessons/13-function-calling.md'